In [1]:
import jax
import jax.numpy as jnp

import lal
import lalsimulation as lalsim
from lalsimulation import SimIMRPhenomXPMSAAngles

from ripplegw.constants import MSUN
from ripplegw.waveforms.LALSimIMRPhenomX import XLALSimIMRPhenomXPMSAAngles 

jax.config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt

/tmp/ipykernel_421440/699506714.py:4: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal
/home/robinc/jax_gw/ripple/src/ripplegw/waveforms/LALSimIMRPhenomX_precession.py:2441: SyntaxWarning: invalid escape sequence '\c'
  """


In [2]:
sampling_frequency = 2048

freqs = jnp.linspace(20.0, sampling_frequency / 2.0, num=sampling_frequency)

m1_si = 30.0 * MSUN
m2_si = 20.0 * MSUN
chi1x = 0.8
chi1y = 0.2
chi1z = 0.4
chi2x = 0.1
chi2y = 0.2
chi2z = 0.3
inclination = 0.0
f_ref_in = 20.0
mprime = 2
lal_params = {
    "IMRPhenomXPrecVersion": 220, 
    "PNRUseTunedAngles": 0,
    "AntisymmetricWaveform": 0,
    "PNRUseTunedCoprec": 0,
    "ExpansionOrder": 5
}

print("Computing MSA angles with ripple...")

alphas_ripple, gammas_ripple, cosbetas_ripple = jax.jit(XLALSimIMRPhenomXPMSAAngles)(
    freqs,
    m1_si,
    m2_si,
    chi1x,
    chi1y,
    chi1z,
    chi2x,
    chi2y,
    chi2z,
    inclination,
    f_ref_in,
    mprime,
    lal_params
)

laldict = lal.CreateDict()
lalsim.SimInspiralWaveformParamsInsertPhenomXPrecVersion(laldict, 220)

print("Computing MSA angles with LAL...")

alphas_lal, gammas_lal, cosbetas_lal = SimIMRPhenomXPMSAAngles(
    freqs,
    m1_si,
    m2_si,
    chi1x,
    chi1y,
    chi1z,
    chi2x,
    chi2y,
    chi2z,
    inclination,
    f_ref_in,
    mprime,
    laldict
)

print("#" * 70)
print("ripple MSA angles")
print(f"alphas: {alphas_ripple}\ngammas: {gammas_ripple}\ncosbetas: {cosbetas_ripple}")
print("#" * 70)
print("lal MSA angles")
print(f"alphas: {alphas_lal.data}\ngammas: {gammas_lal.data}\ncosbetas: {cosbetas_lal.data}")
print("#" * 70)

Computing MSA angles with ripple...


UnexpectedTracerError: Encountered an unexpected tracer. A function transformed by JAX had a side effect, allowing for a reference to an intermediate value with type float64[] wrapped in a DynamicJaxprTracer to escape the scope of the transformation.
JAX transformations require that functions explicitly return their outputs, and disallow saving intermediate values to global state.
The function being traced when the value leaked was flag_222_or_223 at /home/robinc/jax_gw/ripple/src/ripplegw/waveforms/initialise_MSA_system.py:448 traced for cond.
------------------------------
The leaked intermediate value was created on line /home/robinc/jax_gw/ripple/src/ripplegw/waveforms/initialise_MSA_system.py:449:40 (IMRPhenomX_Initialize_MSA_System.<locals>.flag_222_or_223). 
------------------------------
When the value was created, the final 5 stack frames (most recent last) excluding JAX-internal frames were:
------------------------------
<string>:17:2 (__create_fn__.<locals>.__init__)
/home/

In [ ]:
# plt.plot(freqs, alphas_ripple - alphas_lal.data, label=r"$\Delta \alpha$")
# plt.plot(freqs, gammas_ripple - gammas_lal.data, label=r"$\Delta \gamma$")
# plt.plot(freqs, cosbetas_ripple - cosbetas_lal.data, label=r"$\Delta \cos \beta$")
# plt.xlabel("Frequency (Hz)")
# plt.ylabel(r"$\Delta$")
# plt.title("Difference between ripple and LAL MSA angles")
# plt.xscale("log")
# plt.yscale("log")
# plt.legend()
# plt.grid(ls="--")
# plt.show()